In [14]:
import stable_worldmodel as swm
from stable_worldmodel.policy import PlanConfig, WorldModelPolicy
from stable_worldmodel.solver import CEMSolver
from stable_worldmodel.wm.utils import load_pretrained
from random import randint
import torch

In [15]:
dataset = swm.data.load_dataset(
    'tworooms.lance',
    num_steps=5,
    frameskip=1,
    keys_to_load=['pixels', 'action', 'state'],
)

In [16]:
device = 'mps'
model = load_pretrained('tworoomwm/weights.pt').to(device).eval()
model.requires_grad_(False)

num_envs=1

world = swm.World(
    'swm/TwoRoom-v1',
    num_envs=num_envs,
    image_shape=(64, 64),
    max_episode_steps=100,
)

solver = CEMSolver(
    model=model,
    num_samples=300,
    n_steps=5,
    device=device,
)

policy = WorldModelPolicy(
    solver=solver,
    config=PlanConfig(
        horizon=3,
        receding_horizon=3
    ),
)

max_episode = 256
episodes_idx = [randint(0, max_episode-1) for _ in range(num_envs)]
min_episode_length = min([dataset.load_episode(ep)['action'].shape[0] for ep in episodes_idx])

world.set_policy(policy)
results = world.evaluate(
    dataset=dataset,
    seed=0,
    episodes_idx=episodes_idx,
    start_steps=[0 for _ in range(num_envs)],
    goal_offset=min_episode_length-1,
    eval_budget=50,
    video='videos/tworooms'
)
world.close()

print(results)


CEM solve time: 3.7367 seconds
CEM solve time: 3.7267 seconds
CEM solve time: 3.7183 seconds
CEM solve time: 3.7272 seconds
CEM solve time: 3.7211 seconds
CEM solve time: 3.7176 seconds
CEM solve time: 3.7220 seconds
CEM solve time: 3.7249 seconds
CEM solve time: 3.7285 seconds
CEM solve time: 3.7196 seconds
CEM solve time: 3.7238 seconds
CEM solve time: 3.7151 seconds
CEM solve time: 3.7227 seconds
CEM solve time: 3.7238 seconds
CEM solve time: 3.7198 seconds
CEM solve time: 3.7214 seconds


/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/subprocess.py:1885: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = _fork_exec(


CEM solve time: 3.7194 seconds
{'success_rate': 0.0, 'episode_successes': array([False]), 'seeds': None}
